# 🚀 AI Startup Success Predictor using Machine Learning
### Advanced ML Mini Project

**Objective:** Predict whether a startup will be *Successful (Acquired)* or *Failed (Closed)* using
company, funding, team and market-related features, following the complete ML lifecycle:

`Data Collection → Cleaning → EDA → Feature Engineering → Encoding → Scaling → Modeling → Evaluation → Feature Importance → Deployment`

**Dataset:** `startup_data_cleaned_JAS.csv` (923 startups, 47 columns, target = `labels` / `status`)

In [ ]:
# ============================================
# STEP 0: Import Required Libraries
# ============================================
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (8, 5)

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                              roc_auc_score, roc_curve, confusion_matrix, classification_report)

import pickle

# XGBoost is optional -- if it isn't installed, we automatically fall back to
# GradientBoostingClassifier so the notebook NEVER throws an ImportError.
try:
    from xgboost import XGBClassifier
    XGB_AVAILABLE = True
    print("XGBoost is available - using XGBClassifier.")
except ImportError:
    XGB_AVAILABLE = False
    print("XGBoost not found (pip install xgboost to enable it).")
    print("Falling back to GradientBoostingClassifier so the notebook still runs with 0 errors.")

print("All libraries imported successfully ✅")

## Step 1: Data Collection

In [ ]:
# Load the dataset
df = pd.read_csv('startup_data_cleaned_JAS.csv')
print("Dataset loaded successfully ✅")
print("Shape:", df.shape)

In [ ]:
df.head()

## Step 2: Data Understanding

In [ ]:
print("Rows, Columns :", df.shape)
print("\nColumn Data Types:\n")
print(df.dtypes)

In [ ]:
df.info()

In [ ]:
df.describe(include='all').T

In [ ]:
print("Missing values per column:\n")
print(df.isnull().sum()[df.isnull().sum() > 0])
print("\nTotal missing values:", df.isnull().sum().sum())
print("Duplicate rows:", df.duplicated().sum())

## Step 3: Data Cleaning

In [ ]:
# Drop columns that don't help prediction (IDs, names, redundant/raw location fields)
irrelevant_cols = ['id', 'object_id', 'name', 'zip_code', 'state_code.1']
df_clean = df.drop(columns=irrelevant_cols)

# Remove exact duplicate rows (if any)
before = df_clean.shape[0]
df_clean = df_clean.drop_duplicates()
after = df_clean.shape[0]

print(f"Dropped {len(irrelevant_cols)} irrelevant columns.")
print(f"Removed {before - after} duplicate rows.")
print("New shape:", df_clean.shape)

In [ ]:
# Handle outliers in funding_total_usd using the IQR method (cap, don't delete)
Q1 = df_clean['funding_total_usd'].quantile(0.25)
Q3 = df_clean['funding_total_usd'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.boxplot(x=df_clean['funding_total_usd'], ax=axes[0], color='salmon')
axes[0].set_title('Before Outlier Capping')

df_clean['funding_total_usd'] = np.clip(df_clean['funding_total_usd'], lower_bound, upper_bound)

sns.boxplot(x=df_clean['funding_total_usd'], ax=axes[1], color='seagreen')
axes[1].set_title('After Outlier Capping')
plt.tight_layout()
plt.show()

## Step 4: Exploratory Data Analysis (EDA)

In [ ]:
# Target variable distribution: Countplot + Pie Chart
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.countplot(x='labels', data=df_clean, palette='Set2', ax=axes[0])
axes[0].set_xticklabels(['Failed (0)', 'Successful (1)'])
axes[0].set_title('Startup Success vs Failure Count')

df_clean['labels'].value_counts().plot.pie(
    autopct='%1.1f%%', labels=['Successful', 'Failed'],
    colors=['#66c2a5', '#fc8d62'], ax=axes[1])
axes[1].set_ylabel('')
axes[1].set_title('Success Rate Share')

plt.tight_layout()
plt.show()

In [ ]:
# Funding distribution: Histogram + Boxplot by Status
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.histplot(df_clean['funding_total_usd'], bins=30, kde=True, color='steelblue', ax=axes[0])
axes[0].set_title('Distribution of Total Funding Raised')

sns.boxplot(x='labels', y='funding_total_usd', data=df_clean, palette='Set3', ax=axes[1])
axes[1].set_xticklabels(['Failed', 'Successful'])
axes[1].set_title('Funding Amount by Startup Status')

plt.tight_layout()
plt.show()

In [ ]:
# Industry (category) distribution: Bar Chart + Success Rate by Industry
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

df_clean['category_code'].value_counts().head(10).plot(kind='bar', color='coral', ax=axes[0])
axes[0].set_title('Top 10 Industries by Number of Startups')
axes[0].set_xlabel('Industry')
axes[0].set_ylabel('Count')

df_clean.groupby('category_code')['labels'].mean().sort_values(ascending=False).head(10)\
    .plot(kind='bar', color='mediumseagreen', ax=axes[1])
axes[1].set_title('Top 10 Industries by Success Rate')
axes[1].set_xlabel('Industry')
axes[1].set_ylabel('Success Rate')

plt.tight_layout()
plt.show()

In [ ]:
# Relationships (network/team size proxy) & Funding vs Relationships scatter
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.histplot(df_clean['relationships'], bins=20, color='purple', ax=axes[0])
axes[0].set_title('Distribution of Founder/Investor Relationships')

sns.scatterplot(x='funding_total_usd', y='relationships', hue='labels',
                 palette={0: 'red', 1: 'green'}, data=df_clean, ax=axes[1])
axes[1].set_title('Funding vs Relationships (colored by Status)')

plt.tight_layout()
plt.show()

In [ ]:
# Correlation Heatmap of numeric features
numeric_cols = df_clean.select_dtypes(include=[np.number]).columns.tolist()

plt.figure(figsize=(16, 12))
sns.heatmap(df_clean[numeric_cols].corr(), cmap='coolwarm', annot=False, linewidths=0.3)
plt.title('Correlation Heatmap of Numeric Features')
plt.show()

In [ ]:
# Pair Plot of key numeric features vs target
key_features = ['funding_total_usd', 'relationships', 'funding_rounds',
                 'milestones', 'avg_participants', 'labels']
sns.pairplot(df_clean[key_features], hue='labels', palette={0: 'red', 1: 'green'}, diag_kind='kde')
plt.suptitle('Pair Plot of Key Features', y=1.02)
plt.show()

## Step 5: Feature Engineering

In [ ]:
df_fe = df_clean.copy()

# Funding raised per funding round
df_fe['funding_per_round'] = df_fe['funding_total_usd'] / df_fe['funding_rounds'].replace(0, 1)

# How many years the startup spent actively raising funds
df_fe['funding_duration_years'] = (df_fe['age_last_funding_year'] - df_fe['age_first_funding_year']).clip(lower=0)

# Milestone achievement rate over time
df_fe['milestone_rate'] = df_fe['milestones'] / (df_fe['age_last_milestone_year'].abs() + 1)

# Funding raised per relationship (investor/founder connection efficiency)
df_fe['funding_per_relationship'] = df_fe['funding_total_usd'] / df_fe['relationships'].replace(0, 1)

# Diversity of funding round types raised (VC, angel, round A-D)
round_type_cols = ['has_VC', 'has_angel', 'has_roundA', 'has_roundB', 'has_roundC', 'has_roundD']
df_fe['funding_type_diversity'] = df_fe[round_type_cols].sum(axis=1)

print("New engineered features created:")
print(['funding_per_round', 'funding_duration_years', 'milestone_rate',
       'funding_per_relationship', 'funding_type_diversity'])
df_fe[['funding_per_round', 'funding_duration_years', 'milestone_rate',
       'funding_per_relationship', 'funding_type_diversity']].describe()

## Step 6: Data Encoding

In [ ]:
# Label Encode categorical columns
le_category = LabelEncoder()
df_fe['category_code_enc'] = le_category.fit_transform(df_fe['category_code'].astype(str))

le_state = LabelEncoder()
df_fe['state_code_enc'] = le_state.fit_transform(df_fe['state_code'].astype(str))

le_city = LabelEncoder()
df_fe['city_enc'] = le_city.fit_transform(df_fe['city'].astype(str))

# Drop original text/categorical/date columns no longer needed for modeling
cols_to_drop = ['category_code', 'state_code', 'city',
                 'founded_at', 'closed_at', 'first_funding_at', 'last_funding_at', 'status']
df_model = df_fe.drop(columns=cols_to_drop)

print("Final modeling dataset shape:", df_model.shape)
df_model.head()

## Step 7: Feature Scaling

In [ ]:
X = df_model.drop(columns=['labels'])
y = df_model['labels']

scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

X_scaled.describe().T.head()

## Step 8: Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y)

print("Training set shape:", X_train.shape)
print("Testing set shape :", X_test.shape)
print("Train target distribution:\n", y_train.value_counts(normalize=True))

## Step 9: Model Training

We train **4 classification algorithms**: Logistic Regression, Decision Tree, Random Forest, and XGBoost (Gradient Boosting fallback if XGBoost isn't installed).

In [ ]:
# Model 1: Logistic Regression (baseline)
log_reg = LogisticRegression(max_iter=1000, random_state=42)
log_reg.fit(X_train, y_train)
print("Logistic Regression trained ✅")

In [ ]:
# Model 2: Decision Tree
dt_model = DecisionTreeClassifier(max_depth=6, random_state=42)
dt_model.fit(X_train, y_train)
print("Decision Tree trained ✅")

In [ ]:
# Model 3: Random Forest
rf_model = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42)
rf_model.fit(X_train, y_train)
print("Random Forest trained ✅")

In [ ]:
# Model 4: XGBoost (falls back to GradientBoostingClassifier automatically if not installed)
if XGB_AVAILABLE:
    xgb_model = XGBClassifier(n_estimators=200, max_depth=5, learning_rate=0.1,
                               eval_metric='logloss', random_state=42)
else:
    xgb_model = GradientBoostingClassifier(n_estimators=200, max_depth=5,
                                            learning_rate=0.1, random_state=42)

xgb_model.fit(X_train, y_train)
print(f"{'XGBoost' if XGB_AVAILABLE else 'GradientBoosting (XGBoost fallback)'} trained ✅")

## Step 10: Model Evaluation & Comparison

In [ ]:
models = {
    'Logistic Regression': log_reg,
    'Decision Tree': dt_model,
    'Random Forest': rf_model,
    'XGBoost' if XGB_AVAILABLE else 'GradientBoosting': xgb_model
}

results = []
predictions = {}

for name, model in models.items():
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    predictions[name] = (y_pred, y_proba)

    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1 Score': f1_score(y_test, y_pred),
        'ROC-AUC': roc_auc_score(y_test, y_proba)
    })

results_df = pd.DataFrame(results).sort_values(by='F1 Score', ascending=False).reset_index(drop=True)
print("Model evaluation complete ✅")

In [ ]:
results_df.style.background_gradient(cmap='Greens', subset=['Accuracy', 'Precision', 'Recall', 'F1 Score', 'ROC-AUC'])

In [ ]:
# Bar chart comparing all models across all metrics
results_df.set_index('Model')[['Accuracy', 'Precision', 'Recall', 'F1 Score', 'ROC-AUC']].plot(
    kind='bar', figsize=(12, 6), colormap='viridis')
plt.title('Model Performance Comparison')
plt.ylabel('Score')
plt.xticks(rotation=15)
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
# Confusion Matrices for all 4 models
fig, axes = plt.subplots(2, 2, figsize=(11, 9))
for ax, (name, (y_pred, _)) in zip(axes.flatten(), predictions.items()):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
                xticklabels=['Failed', 'Successful'], yticklabels=['Failed', 'Successful'], ax=ax)
    ax.set_title(f'{name} - Confusion Matrix')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
plt.tight_layout()
plt.show()

In [ ]:
# ROC Curve comparison for all models
plt.figure(figsize=(8, 6))
for name, (_, y_proba) in predictions.items():
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc_score = roc_auc_score(y_test, y_proba)
    plt.plot(fpr, tpr, label=f'{name} (AUC = {auc_score:.3f})')

plt.plot([0, 1], [0, 1], 'k--', label='Random Guess')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve Comparison')
plt.legend(loc='lower right')
plt.show()

In [ ]:
best_model_name = results_df.iloc[0]['Model']
best_model = models[best_model_name]
print(f"🏆 Best performing model: {best_model_name}")
print(results_df.iloc[0])
print("\nDetailed Classification Report:\n")
print(classification_report(y_test, predictions[best_model_name][0], target_names=['Failed', 'Successful']))

## Step 11: Feature Importance Analysis

In [ ]:
# Feature importance from Random Forest & XGBoost/GradientBoosting side by side
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

rf_importance = pd.Series(rf_model.feature_importances_, index=X.columns).sort_values(ascending=False).head(12)
rf_importance.plot(kind='barh', color='teal', ax=axes[0])
axes[0].invert_yaxis()
axes[0].set_title('Top 12 Feature Importances - Random Forest')

xgb_importance = pd.Series(xgb_model.feature_importances_, index=X.columns).sort_values(ascending=False).head(12)
xgb_importance.plot(kind='barh', color='darkorange', ax=axes[1])
axes[1].invert_yaxis()
axes[1].set_title(f"Top 12 Feature Importances - {'XGBoost' if XGB_AVAILABLE else 'GradientBoosting'}")

plt.tight_layout()
plt.show()

print("Most influential features (Random Forest):")
print(rf_importance.head(5))

## Step 12: Save the Best Model (for Streamlit Deployment)

In [ ]:
# Persist the best model + scaler + encoders so the Streamlit app can load them
with open('best_startup_model.pkl', 'wb') as f:
    pickle.dump(best_model, f)

with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

with open('encoders.pkl', 'wb') as f:
    pickle.dump({'category': le_category, 'state': le_state, 'city': le_city}, f)

print("Model, scaler and encoders saved successfully ✅")
print("Files created: best_startup_model.pkl, scaler.pkl, encoders.pkl")

## ✅ Conclusion

- All 4 models were trained and evaluated on the cleaned & engineered startup dataset.
- The comparison table and charts above identify the best-performing model based on **F1 Score** and **ROC-AUC**.
- Feature importance analysis shows which business factors (funding, relationships, milestones, industry) most influence startup survival.
- The best model has been saved (`.pkl`) and is ready to be loaded into a **Streamlit** app for deployment, as required by the project brief.